# Product Ranker – Amazon Reviews’23 (.jsonl.gz)

### What this notebook does
1. Loads TF-IDF + Logistic Regression sentiment artifacts
2. Streams Amazon Reviews’23 `.jsonl.gz` review files
3. Runs sentiment inference **per review** (batched for speed)
4. Aggregates review stats per product (`parent_asin`)
5. Stores aggregation in SQLite incrementally (WAL mode)
6. Computes Bayesian smoothed rating + sentiment score
7. Exports results to:
   - CSV (`ml/data/reviews-output/product_ranking.csv`)
   - SQLite (`ml/data/reviews-output/product_ranking.sqlite`)


In [43]:
# =========================
# PRODUCT RANKER PIPELINE
# Streaming .jsonl.gz -> Sentiment -> Product ranking
# =========================

import os
import gzip
import json
import time
import gc
import sqlite3
import hashlib
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
from tqdm.auto import tqdm

# Optional speed-up
try:
    import orjson
    ORJSON = True
except Exception:
    ORJSON = False

print("orjson enabled:", ORJSON)


orjson enabled: True


In [ ]:
# =========================
# CONFIG + ROOT DETECTION
# =========================

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by looking upward for: ml/data/reviews"""
    start = Path.cwd().resolve() if start is None else start.resolve()

    for p in [start] + list(start.parents):
        if (p / "ml" / "data" / "reviews").exists():
            return p

    raise RuntimeError(f"Project root not found above: {start}")

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "ml" / "data"
REVIEWS_DIR = DATA_DIR / "reviews"
OUT_DIR = DATA_DIR / "reviews-output"
MODELS_DIR = PROJECT_ROOT / "ml" / "models"

OUT_DIR.mkdir(parents=True, exist_ok=True)

VECTORIZER_PATH = MODELS_DIR / "sentiment_tfidf_vectorizer.joblib"
MODEL_PATH = MODELS_DIR / "sentiment_logistic_regression.joblib"

SQLITE_PATH = OUT_DIR / "product_ranking.sqlite"
CSV_PATH = OUT_DIR / "product_ranking.csv"

INPUT_GLOB = "*.gz"

# performance tuning for 8GB RAM
BATCH_SIZE_TEXT = 4096
UPSERT_CHUNK = 5000

# Bayesian smoothing
M_RATING = 50
M_SENT = 50

PROCESS_MODE = "incremental"  # "incremental" skips already processed gz files
# PROCESS_MODE = "full"       # "full" reprocess everything

assert REVIEWS_DIR.exists(), f"Missing folder: {REVIEWS_DIR}"
assert VECTORIZER_PATH.exists(), f"Missing: {VECTORIZER_PATH}"
assert MODEL_PATH.exists(), f"Missing: {MODEL_PATH}"

gz_files = sorted(REVIEWS_DIR.glob(INPUT_GLOB))
print("Found gz files:", len(gz_files))
print("Example:", gz_files[0] if gz_files else None)


PROJECT_ROOT: D:\github\git repositories\smart-shopping-search
Found gz files: 10
Example: D:\github\git repositories\smart-shopping-search\ml\data\reviews\All_Beauty.jsonl.gz


In [45]:
# =========================
# LOAD SENTIMENT ARTIFACTS
# =========================

t0 = time.time()
tfidf = joblib.load(VECTORIZER_PATH)
model = joblib.load(MODEL_PATH)
print(f"Loaded artifacts in {time.time()-t0:.2f}s")

print(type(tfidf))
print(type(model))


Loaded artifacts in 0.09s
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'sklearn.linear_model._logistic.LogisticRegression'>


In [46]:
# =========================
# JSONL STREAM HELPERS
# =========================

def parse_json_line(raw: bytes):
    if ORJSON:
        return orjson.loads(raw)
    return json.loads(raw)

def iter_jsonl_gz(path: Path):
    with gzip.open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield parse_json_line(line)
            except Exception:
                continue

def safe_parent_asin(row: dict) -> str:
    pa = row.get("parent_asin")
    if pa:
        return str(pa)
    a = row.get("asin")
    return str(a) if a else ""

def build_text(row: dict) -> str:
    title = row.get("title") or ""
    text = row.get("text") or ""
    return f"{title}. {text}".strip()

def to_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default

def to_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default


In [47]:
# =========================
# SENTIMENT INFERENCE
# =========================

def infer_sentiment_probs(texts: list[str]) -> np.ndarray:
    """Probability of positive sentiment (class=1)."""
    X = tfidf.transform(texts)
    return model.predict_proba(X)[:, 1]


In [48]:
# =========================
# FAST FILE HASH (INCREMENTAL MODE)
# =========================

def fast_file_hash(file_path: str, sample_bytes: int = 1024 * 1024) -> str:
    st = os.stat(file_path)
    size = st.st_size
    mtime = int(st.st_mtime)

    h = hashlib.md5()
    h.update(f"{size}:{mtime}".encode("utf-8"))

    with open(file_path, "rb") as f:
        h.update(f.read(sample_bytes))
        if size > sample_bytes:
            f.seek(max(0, size - sample_bytes))
            h.update(f.read(sample_bytes))

    return h.hexdigest()


In [49]:
# =========================
# SQLITE SETUP
# =========================

def sqlite_connect(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA temp_store=MEMORY;")
    conn.execute("PRAGMA cache_size=-20000;")
    return conn

def sqlite_init(conn):
    conn.execute("""
    CREATE TABLE IF NOT EXISTS product_stats (
        parent_asin TEXT PRIMARY KEY,
        review_count INTEGER,
        sum_rating REAL,
        sum_sent REAL,
        pos_count INTEGER,
        neg_count INTEGER,
        helpful_sum INTEGER,
        last_updated_ts INTEGER
    );
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS processed_files (
        file_name TEXT PRIMARY KEY,
        file_hash TEXT NOT NULL,
        review_lines INTEGER NOT NULL,
        processed_at TEXT NOT NULL
    );
    """)

    conn.commit()

def is_file_processed(conn, file_name: str, file_hash: str) -> bool:
    cur = conn.cursor()
    cur.execute(
        "SELECT 1 FROM processed_files WHERE file_name=? AND file_hash=? LIMIT 1",
        (file_name, file_hash)
    )
    return cur.fetchone() is not None

def mark_file_processed(conn, file_name: str, file_hash: str, review_lines: int):
    cur = conn.cursor()
    cur.execute("""
    INSERT INTO processed_files (file_name, file_hash, review_lines, processed_at)
    VALUES (?, ?, ?, ?)
    ON CONFLICT(file_name) DO UPDATE SET
        file_hash=excluded.file_hash,
        review_lines=excluded.review_lines,
        processed_at=excluded.processed_at
    """, (file_name, file_hash, review_lines, datetime.utcnow().isoformat()))
    conn.commit()

def upsert_products(conn, rows):
    conn.executemany("""
    INSERT INTO product_stats
    (parent_asin, review_count, sum_rating, sum_sent, pos_count, neg_count, helpful_sum, last_updated_ts)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(parent_asin) DO UPDATE SET
        review_count = product_stats.review_count + excluded.review_count,
        sum_rating   = product_stats.sum_rating + excluded.sum_rating,
        sum_sent     = product_stats.sum_sent + excluded.sum_sent,
        pos_count    = product_stats.pos_count + excluded.pos_count,
        neg_count    = product_stats.neg_count + excluded.neg_count,
        helpful_sum  = product_stats.helpful_sum + excluded.helpful_sum,
        last_updated_ts = excluded.last_updated_ts;
    """, rows)


In [50]:
# =========================
# RUN PIPELINE
# =========================

def run_pipeline(files: list[Path]):
    conn = sqlite_connect(SQLITE_PATH)
    sqlite_init(conn)

    agg = defaultdict(lambda: {
        "n": 0,
        "sum_rating": 0.0,
        "sum_sent": 0.0,
        "pos": 0,
        "neg": 0,
        "helpful": 0
    })

    batch_texts, batch_asins, batch_ratings, batch_helpful = [], [], [], []

    processed_files = 0
    skipped_files = 0
    total_reviews = 0

    start = time.time()

    def flush(ts: int):
        if not agg:
            return 0

        items = []
        for asin, s in agg.items():
            items.append((
                asin,
                s["n"],
                s["sum_rating"],
                s["sum_sent"],
                s["pos"],
                s["neg"],
                s["helpful"],
                ts
            ))

        for i in range(0, len(items), UPSERT_CHUNK):
            upsert_products(conn, items[i:i+UPSERT_CHUNK])
            conn.commit()

        agg.clear()
        return len(items)

    for fp in files:
        file_path = str(fp)
        file_name = fp.name
        file_hash = fast_file_hash(file_path)

        if PROCESS_MODE == "incremental" and is_file_processed(conn, file_name, file_hash):
            skipped_files += 1
            continue

        print(f"\nProcessing: {file_name}")
        processed_files += 1
        file_lines = 0

        for row in tqdm(iter_jsonl_gz(fp), desc=file_name):
            asin = safe_parent_asin(row)
            if not asin:
                continue

            rating = to_float(row.get("rating") or row.get("stars"), 0.0)
            helpful = to_int(row.get("helpful_vote"), 0)
            txt = build_text(row)

            if not txt.strip():
                continue

            batch_texts.append(txt)
            batch_asins.append(asin)
            batch_ratings.append(rating)
            batch_helpful.append(helpful)

            file_lines += 1
            total_reviews += 1

            if len(batch_texts) >= BATCH_SIZE_TEXT:
                probs = infer_sentiment_probs(batch_texts)

                for a, r, hv, p in zip(batch_asins, batch_ratings, batch_helpful, probs):
                    s = agg[a]
                    s["n"] += 1
                    s["sum_rating"] += float(r)
                    s["sum_sent"] += float(p)
                    s["helpful"] += int(hv)
                    if p >= 0.5:
                        s["pos"] += 1
                    else:
                        s["neg"] += 1

                batch_texts.clear()
                batch_asins.clear()
                batch_ratings.clear()
                batch_helpful.clear()

                if len(agg) > 250_000:
                    ts = int(time.time())
                    flushed = flush(ts)
                    print("Safety flush products:", flushed)
                    gc.collect()

        # flush remainder batch
        if batch_texts:
            probs = infer_sentiment_probs(batch_texts)
            for a, r, hv, p in zip(batch_asins, batch_ratings, batch_helpful, probs):
                s = agg[a]
                s["n"] += 1
                s["sum_rating"] += float(r)
                s["sum_sent"] += float(p)
                s["helpful"] += int(hv)
                if p >= 0.5:
                    s["pos"] += 1
                else:
                    s["neg"] += 1

            batch_texts.clear()
            batch_asins.clear()
            batch_ratings.clear()
            batch_helpful.clear()

        ts = int(time.time())
        flushed = flush(ts)
        mark_file_processed(conn, file_name, file_hash, file_lines)
        print(f"File done: {file_name} reviews={file_lines:,} products_flushed={flushed:,}")

        gc.collect()

    ts = int(time.time())
    flushed = flush(ts)
    print("Final flush products:", flushed)

    conn.close()

    elapsed = time.time() - start
    print("\n=== PIPELINE DONE ===")
    print("Processed gz files:", processed_files)
    print("Skipped gz files  :", skipped_files)
    print("Total reviews used:", total_reviews)
    print(f"Elapsed: {elapsed/60:.2f} minutes")


In [51]:
# =========================
# RANKING COMPUTATION
# =========================

def compute_ranking() -> pd.DataFrame:
    conn = sqlite_connect(SQLITE_PATH)
    df = pd.read_sql_query("""
        SELECT parent_asin, review_count, sum_rating, sum_sent, pos_count, neg_count, helpful_sum
        FROM product_stats
    """, conn)
    conn.close()

    print("Products loaded:", len(df))
    if df.empty:
        return df

    df["avg_rating"] = df["sum_rating"] / df["review_count"].clip(lower=1)
    df["avg_sent"] = df["sum_sent"] / df["review_count"].clip(lower=1)

    C_rating = df["avg_rating"].mean()
    C_sent = df["avg_sent"].mean()

    v = df["review_count"].astype(float)
    R = df["avg_rating"]
    S = df["avg_sent"]

    df["bayes_rating"] = (v/(v+M_RATING))*R + (M_RATING/(v+M_RATING))*C_rating
    df["bayes_sent"]   = (v/(v+M_SENT))*S + (M_SENT/(v+M_SENT))*C_sent

    df["helpful_w"] = np.log1p(df["helpful_sum"].clip(lower=0))
    df["helpful_w_norm"] = df["helpful_w"] / df["helpful_w"].max()

    df["final_score"] = (
        0.65*df["bayes_rating"] +
        0.35*(df["bayes_sent"]*5.0) +
        0.10*df["helpful_w_norm"]
    )

    df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df)+1)

    return df


In [52]:
# =========================
# EXECUTE END TO END
# =========================

run_pipeline(gz_files)
rank_df = compute_ranking()

rank_df.to_csv(CSV_PATH, index=False)
print("Saved ranking CSV:", CSV_PATH)

rank_df.head(20)



Processing: All_Beauty.jsonl.gz


All_Beauty.jsonl.gz: 701528it [00:25, 27006.13it/s]
C:\temp\ipykernel_33628\1067010463.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  """, (file_name, file_hash, review_lines, datetime.utcnow().isoformat()))


File done: All_Beauty.jsonl.gz reviews=701,528 products_flushed=112,565

Processing: Amazon_Fashion.jsonl.gz


Amazon_Fashion.jsonl.gz: 421888it [00:18, 4078.42it/s] 

Safety flush products: 251177


...

Cell_Phones_and_Accessories.jsonl.gz: 20701184it [16:26, 1384.50it/s] 

Safety flush products: 250029


Cell_Phones_and_Accessories.jsonl.gz: 20812945it [16:30, 21022.84it/s]


File done: Cell_Phones_and_Accessories.jsonl.gz reviews=20,812,945 products_flushed=51,256
Final flush products: 0

=== PIPELINE DONE ===
Processed gz files: 10
Skipped gz files  : 0
Total reviews used: 119309225
Elapsed: 123.09 minutes
Products loaded: 11519633
Saved ranking CSV: D:\github\git repositories\smart-shopping-search\ml\data\reviews-output\product_ranking.csv


,parent_asin,review_count,sum_rating,sum_sent,pos_count,neg_count,helpful_sum,avg_rating,avg_sent,bayes_rating,bayes_sent,helpful_w,helpful_w_norm,final_score,rank
0,0062986511,1459,7176.0,1391.075749,1443,16,1008,4.918437,0.953445,4.893754,0.944902,6.916715,0.585968,4.893116,1
1,0578891743,458,2287.0,423.179498,449,9,865,4.993450,0.923973,4.912745,0.901499,6.763885,0.573020,4.828209,2
2,0063008203,434,2141.0,411.481873,422,12,566,4.933180,0.948115,4.854699,0.922032,6.340359,0.537140,4.822825,3
3,0763622281,2143,10544.0,1934.260237,2058,85,1432,4.920205,0.902595,4.903180,0.897876,7.267525,0.615688,4.819919,4
4,1941698026,534,2657.0,492.964364,528,6,317,4.975655,0.923154,4.906977,0.903675,5.762051,0.488147,4.819781,5
5,0544568036,1656,8165.0,1498.837443,1598,58,693,4.930556,0.905095,4.908367,0.898956,6.542472,0.554263,4.819038,6
6,1465475621,820,4012.0,760.452556,799,21,1190,4.892683,0.927381,4.851350,0.914062,7.082549,0.600017,4.812988,7
7,0061906220,1326,6527.0,1201.230808,1287,39,1017,4.922323,0.905906,4.895112,0.898265,6.925595,0.586720,4.812459,8
8,1452156182,780,3811.0,724.503544,753,27,1307,4.885897,0.928851,4.842981,0.914802,7.176255,0.607955,4.809636,9
9,B08ZW38F5V,1148,5721.0,1003.131801,1070,78,6772,4.983449,0.873808,4.949645,0.866372,8.820699,0.747269,4.808147,10


In [ ]:
import sqlite3

conn = sqlite3.connect(SQLITE_PATH)

rank_df.rename(columns={
    "parent_asin": "product_id",
    "final_score": "review_score"
}).to_sql(
    "product_ranking",
    conn,
    if_exists="replace",
    index=False
)

conn.close()
